# DeepSeek Chat — learning notebook

DeepSeek's API is OpenAI-compatible, so we use the `openai` SDK with `base_url="https://api.deepseek.com"`.

Put your key in a `.env` file next to this notebook:
```
DEEPSEEK_API_KEY=sk-...
```

In [ ]:
%pip install -q openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com",
)
MODEL = "deepseek-chat"  # or "deepseek-reasoner" for step-by-step reasoning

## 1. Single question

In [ ]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful tutor."},
        {"role": "user", "content": "Explain what an API is in 3 sentences."},
    ],
)
print(resp.choices[0].message.content)
print(resp.usage)

## 2. Multi-turn chat (keeps history)

In [ ]:
history = [{"role": "system", "content": "You are a patient tutor. Keep answers short."}]

def chat(user_msg: str) -> str:
    history.append({"role": "user", "content": user_msg})
    resp = client.chat.completions.create(model=MODEL, messages=history)
    reply = resp.choices[0].message.content
    history.append({"role": "assistant", "content": reply})
    return reply

print(chat("What is a Python list?"))

In [ ]:
print(chat("Give me an example of what you just explained."))

## 3. Streaming (prints tokens as they arrive)

In [ ]:
stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Write a haiku about learning to code."}],
    stream=True,
)
for chunk in stream:
    print(chunk.choices[0].delta.content or "", end="", flush=True)